In [1]:
import os, json
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

json_dir = "cve_data"

# ── helpers ────────────────────────────────────────────────────────────────────

def parse_cpe(cpe_str):
    """
    Parse CPE 2.3 URI into (cpe_type, vendor, product, version).

    CPE 2.3 format:
        cpe:2.3:<part>:<vendor>:<product>:<version>:<update>:<edition>:...
        part → 'a' (application), 'o' (operating system), 'h' (hardware)
    """
    parts = cpe_str.split(":")
    part_map = {"a": "Application", "o": "OS", "h": "Hardware"}
    cpe_type = part_map.get(parts[2], parts[2]) if len(parts) > 2 else "N/A"
    vendor   = parts[3].replace("_", " ").title() if len(parts) > 3 else "N/A"
    product  = parts[4].replace("_", " ").title() if len(parts) > 4 else "N/A"
    version  = parts[5] if len(parts) > 5 else "N/A"
    return cpe_type, vendor, product, version


def build_version_range(cpe_match: dict, base_version: str) -> str:
    """
    Build a human-readable version range string from CPE match fields.

    NVD stores affected version ranges using four optional fields:
      - versionStartIncluding  → >= this version
      - versionStartExcluding  → >  this version
      - versionEndIncluding    → <= this version
      - versionEndExcluding    → <  this version

    When none of these are present and the CPE version is not '*' or '-',
    the CPE itself pins an exact version.
    """
    v_si = cpe_match.get("versionStartIncluding")
    v_sx = cpe_match.get("versionStartExcluding")
    v_ei = cpe_match.get("versionEndIncluding")
    v_ex = cpe_match.get("versionEndExcluding")

    lower = f">= {v_si}" if v_si else (f"> {v_sx}" if v_sx else None)
    upper = f"<= {v_ei}" if v_ei else (f"< {v_ex}" if v_ex else None)

    if lower and upper:
        return f"{lower}, {upper}"
    if lower:
        return lower
    if upper:
        return upper

    # Fall back to the version baked into the CPE string
    if base_version not in ("*", "-", "N/A", ""):
        return f"= {base_version}"

    return "All versions"


def extract_cvss(metrics):
    m = {}
    for key, ver_label in [("cvssMetricV31", "3.1"),
                            ("cvssMetricV30", "3.0")]:
        if key in metrics:
            raw = metrics[key][0]
            d   = raw["cvssData"]
            m.update({
                "CVSS Version": ver_label,
                "baseScore": d.get("baseScore"),
                "baseSeverity": d.get("baseSeverity"),
                "attackVector": d.get("attackVector"),
                "attackComplexity": d.get("attackComplexity"),
                "privilegesRequired": d.get("privilegesRequired"),
                "userInteraction": d.get("userInteraction"),
                "scope": d.get("scope"),
                "confidentialityImpact": d.get("confidentialityImpact"),
                "integrityImpact": d.get("integrityImpact"),
                "availabilityImpact": d.get("availabilityImpact"),
                "exploitabilityScore": raw.get("exploitabilityScore"),
                "impactScore": raw.get("impactScore"),
                "vectorString": d.get("vectorString"),
            })
            return m

    if "cvssMetricV2" in metrics:
        raw = metrics["cvssMetricV2"][0]
        d   = raw["cvssData"]
        m.update({
            "CVSS Version": "2.0",
            "baseScore": d.get("baseScore"),
            "baseSeverity": raw.get("baseSeverity"),
            "attackVector": d.get("accessVector"),
            "attackComplexity": d.get("accessComplexity"),
            "privilegesRequired": d.get("authentication"),
            "userInteraction": "REQUIRED" if raw.get("userInteractionRequired") else "NONE",
            "scope": "N/A",
            "confidentialityImpact": d.get("confidentialityImpact"),
            "integrityImpact": d.get("integrityImpact"),
            "availabilityImpact": d.get("availabilityImpact"),
            "exploitabilityScore": raw.get("exploitabilityScore"),
            "impactScore": raw.get("impactScore"),
            "vectorString": d.get("vectorString"),
        })
    return m


# ── parse all JSON files ───────────────────────────────────────────────────────

main_rows, config_rows, ref_rows = [], [], []

for fname in sorted(os.listdir(json_dir)):
    if not fname.endswith(".json"):
        continue
    with open(os.path.join(json_dir, fname), encoding="utf-8") as f:
        data = json.load(f)

    cve_id = data["id"]

    # English description
    desc = next(
        (d["value"] for d in data.get("descriptions", []) if d.get("lang") == "en"),
        "N/A"
    )

    # Weaknesses (CWE)
    weak_primary = weak_secondary = "N/A"
    for w in data.get("weaknesses", []):
        val = next(
            (d["value"] for d in w.get("description", []) if d.get("lang") == "en"),
            None
        )
        if val:
            if w.get("type") == "Primary":
                weak_primary = val
            else:
                weak_secondary = val

    cvss = extract_cvss(data.get("metrics", {}))

    main_rows.append({
        "CVE ID": cve_id,
        "Published": data.get("published", "")[:10],
        "Last Modified": data.get("lastModified", "")[:10],
        "Status": data.get("vulnStatus", "N/A"),
        "Source": data.get("sourceIdentifier", "N/A"),
        "Description": desc,
        "Primary Weakness (CWE)": weak_primary,
        "Secondary Weakness (CWE)": weak_secondary,
        "CVSS Version": cvss.get("CVSS Version"),
        "Base Score": cvss.get("baseScore"),
        "Severity": cvss.get("baseSeverity"),
        "Attack Vector": cvss.get("attackVector"),
        "Attack Complexity": cvss.get("attackComplexity"),
        "Privileges Required": cvss.get("privilegesRequired"),
        "User Interaction": cvss.get("userInteraction"),
        "Scope": cvss.get("scope"),
        "Confidentiality Impact": cvss.get("confidentialityImpact"),
        "Integrity Impact": cvss.get("integrityImpact"),
        "Availability Impact": cvss.get("availabilityImpact"),
        "Exploitability Score": cvss.get("exploitabilityScore"),
        "Impact Score": cvss.get("impactScore"),
        "Vector String": cvss.get("vectorString"),
    })

    # ── Configurations → Affected Components ──────────────────────────────────
    #
    # NVD structures configurations as a list of "configurations".
    # Each configuration has "nodes", which can be nested with AND/OR logic.
    # Each node contains "cpeMatch" entries — one per specific product/version.
    #
    # We walk every node and every cpeMatch entry, capturing:
    #   • The node's boolean operator (AND / OR) — tells you whether ALL
    #     products in the node must be present (AND) or just one (OR).
    #   • Whether the node is negated (rare, but meaningful).
    #   • The cpeMatch fields:
    #       – criteria          : full CPE 2.3 URI
    #       – matchCriteriaId   : NVD's internal UUID for the criteria
    #       – vulnerable        : True → the component IS affected
    #       – versionStart/End  : version range bounds (see build_version_range)
    #
    for cfg_idx, cfg in enumerate(data.get("configurations", []), 1):
        for node_idx, node in enumerate(cfg.get("nodes", []), 1):
            node_operator = node.get("operator", "OR")   # AND | OR
            node_negate   = node.get("negate", False)    # True → invert match

            for cpe in node.get("cpeMatch", []):
                criteria    = cpe.get("criteria", "")
                cpe_type, vendor, product, base_ver = parse_cpe(criteria)
                version_range = build_version_range(cpe, base_ver)
                vulnerable    = cpe.get("vulnerable", False)

                config_rows.append({
                    "CVE ID": cve_id,
                    # --- node context ---
                    "Config #": cfg_idx,
                    "Node #": node_idx,
                    "Node Operator": node_operator,
                    "Node Negated": "YES" if node_negate else "no",
                    # --- component identity ---
                    "CPE Type": cpe_type,           # Application / OS / Hardware
                    "Vendor": vendor,
                    "Product / Component": product,
                    # --- version detail ---
                    "Exact Version (CPE)": base_ver,
                    "Affected Version Range": version_range,
                    "Version Start Incl.": cpe.get("versionStartIncluding", ""),
                    "Version Start Excl.": cpe.get("versionStartExcluding", ""),
                    "Version End Incl.": cpe.get("versionEndIncluding", ""),
                    "Version End Excl.": cpe.get("versionEndExcluding", ""),
                    # --- flags ---
                    "Vulnerable": vulnerable,       # kept as bool for analytics
                    "Match Criteria ID": cpe.get("matchCriteriaId", ""),
                    "Full CPE Criteria": criteria,
                })

    # References
    seen_urls: set = set()
    for ref in data.get("references", []):
        url = ref.get("url", "")
        if url in seen_urls:
            continue
        seen_urls.add(url)
        ref_rows.append({
            "CVE ID": cve_id,
            "URL": url,
            "Tags": ", ".join(ref.get("tags", [])),
        })

df_main   = pd.DataFrame(main_rows)
df_config = pd.DataFrame(config_rows)
df_ref    = pd.DataFrame(ref_rows)

# ── styling constants ──────────────────────────────────────────────────────────

C_HEADER_BG = "1A2B4A"
C_HEADER_FG = "FFFFFF"
C_ROW_ALT   = "F0F4F8"
C_BORDER    = "C8D0DC"

SEVERITY_COLORS = {
    "CRITICAL": "7B0000",
    "HIGH":     "C0392B",
    "MEDIUM":   "D4860A",
    "LOW":      "2E7D32",
    "NONE":     "558B2F",
}

thin   = Side(border_style="thin", color=C_BORDER)
border = Border(left=thin, right=thin, top=thin, bottom=thin)


def style_header(cell, bg=C_HEADER_BG, fg=C_HEADER_FG):
    cell.font      = Font(bold=True, color=fg, name="Calibri", size=10)
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border    = border


def style_data(cell, alt=False):
    cell.font      = Font(name="Calibri", size=10)
    if alt:
        cell.fill  = PatternFill("solid", fgColor=C_ROW_ALT)
    cell.alignment = Alignment(vertical="top", wrap_text=True)
    cell.border    = border


def write_sheet(ws, df, col_widths=None):
    ws.freeze_panes     = "A2"
    ws.auto_filter.ref  = ws.dimensions
    headers = list(df.columns)
    for ci, h in enumerate(headers, 1):
        style_header(ws.cell(row=1, column=ci, value=h))
    for ri, row in enumerate(df.itertuples(index=False), 2):
        alt = ri % 2 == 0
        for ci, val in enumerate(row, 1):
            style_data(ws.cell(row=ri, column=ci, value=val), alt)
    ws.row_dimensions[1].height = 28
    if col_widths:
        for col, w in col_widths.items():
            ws.column_dimensions[col].width = w
    else:
        for col in ws.columns:
            length = max((len(str(c.value or "")) for c in col), default=0)
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(length + 2, 10), 50)


# ── build workbook ─────────────────────────────────────────────────────────────

wb = Workbook()

# ── Sheet 1: CVE Summary ───────────────────────────────────────────────────────
ws1 = wb.active
ws1.title = "CVE Summary"
write_sheet(ws1, df_main, col_widths={
    "A": 18, "B": 12, "C": 14, "D": 14, "E": 28, "F": 60,
    "G": 20, "H": 22, "I": 10, "J": 10, "K": 10, "L": 15,
    "M": 16, "N": 18, "O": 16, "P": 10, "Q": 18, "R": 16,
    "S": 18, "T": 16, "U": 16, "V": 35,
})

# Colour-code the Severity column
sev_col_idx = df_main.columns.get_loc("Severity") + 1
for ri in range(2, len(df_main) + 2):
    cell = ws1.cell(row=ri, column=sev_col_idx)
    sev  = str(cell.value or "").upper()
    if sev in SEVERITY_COLORS:
        cell.fill = PatternFill("solid", fgColor=SEVERITY_COLORS[sev])
        cell.font = Font(bold=True, color="FFFFFF", name="Calibri", size=10)

# Green→Yellow→Red colour scale on Base Score
score_col   = get_column_letter(df_main.columns.get_loc("Base Score") + 1)
score_range = f"{score_col}2:{score_col}{len(df_main) + 1}"
ws1.conditional_formatting.add(score_range,
    ColorScaleRule(
        start_type="num", start_value=0,  start_color="63BE7B",
        mid_type="num",   mid_value=5,    mid_color="FFEB84",
        end_type="num",   end_value=10,   end_color="F8696B",
    )
)

# ── Sheet 2: Affected Components (full CPE detail) ─────────────────────────────
#
# This sheet now includes every field from every cpeMatch entry,
# including all four version-range bounds and the human-readable
# "Affected Version Range" summary column.
#
ws2 = wb.create_sheet("Affected Components")

# Prepare display copy: convert Vulnerable bool → "YES" / "no"
df_config_display = df_config.copy()
df_config_display["Vulnerable"] = df_config_display["Vulnerable"].apply(
    lambda v: "YES" if v is True else "no"
)

write_sheet(ws2, df_config_display, col_widths={
    "A": 18,   # CVE ID
    "B": 10,   # Config #
    "C": 9,    # Node #
    "D": 14,   # Node Operator
    "E": 13,   # Node Negated
    "F": 14,   # CPE Type
    "G": 20,   # Vendor
    "H": 30,   # Product / Component
    "I": 16,   # Exact Version (CPE)
    "J": 36,   # Affected Version Range  ← key new column
    "K": 20,   # Version Start Incl.
    "L": 20,   # Version Start Excl.
    "M": 18,   # Version End Incl.
    "N": 18,   # Version End Excl.
    "O": 12,   # Vulnerable
    "P": 38,   # Match Criteria ID
    "Q": 60,   # Full CPE Criteria
})

# Highlight vulnerable rows and the version range cell
vuln_col_idx = list(df_config_display.columns).index("Vulnerable") + 1
vr_col_idx   = list(df_config_display.columns).index("Affected Version Range") + 1

for ri in range(2, len(df_config_display) + 2):
    vuln_cell = ws2.cell(row=ri, column=vuln_col_idx)
    vr_cell   = ws2.cell(row=ri, column=vr_col_idx)
    if vuln_cell.value == "YES":
        vuln_cell.fill = PatternFill("solid", fgColor="FDECEA")
        vuln_cell.font = Font(bold=True, color="C0392B", name="Calibri", size=10)
        # Also highlight the version range cell to draw attention
        vr_cell.fill   = PatternFill("solid", fgColor="FFF3CD")
        vr_cell.font   = Font(bold=True, name="Calibri", size=10)

# ── Sheet 3: References ────────────────────────────────────────────────────────
ws3 = wb.create_sheet("References")
write_sheet(ws3, df_ref, col_widths={"A": 18, "B": 65, "C": 40})

# ── Sheet 4: Analytics ─────────────────────────────────────────────────────────
ws4 = wb.create_sheet("Analytics")
ws4.sheet_view.showGridLines = False


def h1(ws, row, col, text):
    c = ws.cell(row=row, column=col, value=text)
    c.font      = Font(bold=True, color=C_HEADER_FG, name="Calibri", size=12)
    c.fill      = PatternFill("solid", fgColor=C_HEADER_BG)
    c.alignment = Alignment(horizontal="left", vertical="center")
    c.border    = border
    return c


def kv(ws, row, lbl, val, col=1):
    lc = ws.cell(row=row, column=col, value=lbl)
    lc.font      = Font(bold=True, name="Calibri", size=10)
    lc.fill      = PatternFill("solid", fgColor="E8EDF3")
    lc.border    = border
    lc.alignment = Alignment(horizontal="left", vertical="center")
    vc = ws.cell(row=row, column=col + 1, value=val)
    vc.font      = Font(name="Calibri", size=10)
    vc.border    = border
    vc.alignment = Alignment(horizontal="center", vertical="center")
    return vc


h1(ws4, 1, 1, "📊  CVE Dataset — Analytics Overview")
ws4.merge_cells("A1:F1")
ws4.row_dimensions[1].height = 30

# Use the original boolean column for accurate counts
vuln_df = df_config[df_config["Vulnerable"] == True]

kv(ws4, 3, "Total CVEs",                  len(df_main))
kv(ws4, 4, "Total Vulnerable Components", len(vuln_df))
kv(ws4, 5, "Total References",            len(df_ref))
kv(ws4, 6, "Date Range (Published)",
   f"{df_main['Published'].min()}  →  {df_main['Published'].max()}")

# Severity breakdown
sev_counts = df_main["Severity"].value_counts()
h1(ws4, 9, 1, "Severity Breakdown")
ws4.merge_cells("A9:B9")
for i, (sev, cnt) in enumerate(sev_counts.items(), 10):
    bg = SEVERITY_COLORS.get(str(sev).upper(), "888888")
    for col_i, val in [(1, sev), (2, cnt)]:
        c = ws4.cell(row=i, column=col_i, value=val)
        c.fill      = PatternFill("solid", fgColor=bg)
        c.font      = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
        c.border    = border
        c.alignment = Alignment(horizontal="center", vertical="center")

# Attack Vector breakdown
av_counts = df_main["Attack Vector"].value_counts()
h1(ws4, 9, 4, "Attack Vector Breakdown")
ws4.merge_cells("D9:E9")
for i, (av, cnt) in enumerate(av_counts.items(), 10):
    for col_i, val in [(4, av), (5, cnt)]:
        c = ws4.cell(row=i, column=col_i, value=val)
        c.fill      = PatternFill("solid", fgColor="2C3E50")
        c.font      = Font(color="FFFFFF", name="Calibri", size=10, bold=True)
        c.border    = border
        c.alignment = Alignment(horizontal="center", vertical="center")

# CPE Type breakdown (new — shows Application vs OS vs Hardware split)
cpe_type_counts = vuln_df["CPE Type"].value_counts()
h1(ws4, 9, 7, "CPE Type (Vulnerable)")
ws4.merge_cells("G9:H9")
CPE_COLORS = {"Application": "1565C0", "OS": "6A1B9A", "Hardware": "2E7D32"}
for i, (ct, cnt) in enumerate(cpe_type_counts.items(), 10):
    bg = CPE_COLORS.get(ct, "455A64")
    for col_i, val in [(7, ct), (8, cnt)]:
        c = ws4.cell(row=i, column=col_i, value=val)
        c.fill      = PatternFill("solid", fgColor=bg)
        c.font      = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
        c.border    = border
        c.alignment = Alignment(horizontal="center", vertical="center")

# Top affected vendors
vendor_counts = vuln_df["Vendor"].value_counts().head(10)
h1(ws4, 20, 1, "Top Affected Vendors (vulnerable components)")
ws4.merge_cells("A20:C20")
for i, (v, cnt) in enumerate(vendor_counts.items(), 21):
    c_lbl = ws4.cell(row=i, column=1, value=v)
    c_cnt = ws4.cell(row=i, column=2, value=cnt)
    c_lbl.font      = Font(name="Calibri", size=10)
    c_cnt.font      = Font(name="Calibri", size=10, bold=True)
    c_cnt.fill      = PatternFill("solid", fgColor="DDEEFF")
    for c in (c_lbl, c_cnt):
        c.border    = border
        c.alignment = Alignment(horizontal="left", vertical="center")

# Score statistics
h1(ws4, 20, 4, "Score Statistics")
ws4.merge_cells("D20:E20")
scores = df_main["Base Score"].dropna()
for i, (lbl, val) in enumerate([
    ("Average Base Score", round(scores.mean(),   2)),
    ("Max Base Score",     scores.max()),
    ("Min Base Score",     scores.min()),
    ("Median Base Score",  scores.median()),
], 21):
    lc = ws4.cell(row=i, column=4, value=lbl)
    lc.border    = border
    lc.font      = Font(name="Calibri", size=10)
    lc.alignment = Alignment(horizontal="left")
    vc = ws4.cell(row=i, column=5, value=val)
    vc.border    = border
    vc.font      = Font(bold=True, name="Calibri", size=10)
    vc.alignment = Alignment(horizontal="center")

# Top affected products (new)
product_counts = vuln_df["Product / Component"].value_counts().head(10)
h1(ws4, 20, 7, "Top Affected Products")
ws4.merge_cells("G20:H20")
for i, (p, cnt) in enumerate(product_counts.items(), 21):
    c_lbl = ws4.cell(row=i, column=7, value=p)
    c_cnt = ws4.cell(row=i, column=8, value=cnt)
    c_lbl.font      = Font(name="Calibri", size=10)
    c_cnt.font      = Font(name="Calibri", size=10, bold=True)
    c_cnt.fill      = PatternFill("solid", fgColor="E8F5E9")
    for c in (c_lbl, c_cnt):
        c.border    = border
        c.alignment = Alignment(horizontal="left", vertical="center")

for col, w in [("A", 28), ("B", 12), ("C", 12),
               ("D", 32), ("E", 12), ("F", 4),
               ("G", 28), ("H", 12)]:
    ws4.column_dimensions[col].width = w

# ── save ───────────────────────────────────────────────────────────────────────

out_path = "cve_data.xlsx"
wb.save(out_path)
print("Saved:", out_path)
print(f"CVEs:        {len(df_main)}")
print(f"Components:  {len(df_config)}  ({len(vuln_df)} vulnerable)")
print(f"References:  {len(df_ref)}")

Saved: cve_data.xlsx
CVEs:        10
Components:  46  (12 vulnerable)
References:  10


In [2]:
df_config

,CVE ID,Config #,Node #,Node Operator,Node Negated,CPE Type,Vendor,Product / Component,Exact Version (CPE),Affected Version Range,Version Start Incl.,Version Start Excl.,Version End Incl.,Version End Excl.,Vulnerable,Match Criteria ID,Full CPE Criteria
0,CVE-2014-5431,1,1,OR,no,OS,Baxter,Sigma Spectrum Infusion System Firmware,6.05,= 6.05,,,,,True,E6B02D87-6C75-4D60-8D4C-84628757214F,cpe:2.3:o:baxter:sigma_spectrum_infusion_syste...
1,CVE-2014-5431,1,2,OR,no,Hardware,Baxter,Sigma Spectrum Infusion System,-,All versions,,,,,False,49E25260-EC14-4E98-A86B-CBBE47E26AE5,cpe:2.3:h:baxter:sigma_spectrum_infusion_syste...
2,CVE-2014-5431,1,2,OR,no,Hardware,Baxter,Wireless Battery Module,16,= 16,,,,,False,175BDC61-5163-4D78-BF02-6B9EF5D38841,cpe:2.3:h:baxter:wireless_battery_module:16:*:...
3,CVE-2014-5432,1,1,OR,no,OS,Baxter,Sigma Spectrum Infusion System Firmware,6.05,= 6.05,,,,,True,E6B02D87-6C75-4D60-8D4C-84628757214F,cpe:2.3:o:baxter:sigma_spectrum_infusion_syste...
4,CVE-2014-5432,1,2,OR,no,Hardware,Baxter,Sigma Spectrum Infusion System,-,All versions,,,,,False,49E25260-EC14-4E98-A86B-CBBE47E26AE5,cpe:2.3:h:baxter:sigma_spectrum_infusion_syste...
5,CVE-2014-5432,1,2,OR,no,Hardware,Baxter,Wireless Battery Module,16,= 16,,,,,False,175BDC61-5163-4D78-BF02-6B9EF5D38841,cpe:2.3:h:baxter:wireless_battery_module:16:*:...
6,CVE-2014-5433,1,1,OR,no,OS,Baxter,Sigma Spectrum Infusion System Firmware,6.05,= 6.05,,,,,True,E6B02D87-6C75-4D60-8D4C-84628757214F,cpe:2.3:o:baxter:sigma_spectrum_infusion_syste...
7,CVE-2014-5433,1,2,OR,no,Hardware,Baxter,Sigma Spectrum Infusion System,-,All versions,,,,,False,49E25260-EC14-4E98-A86B-CBBE47E26AE5,cpe:2.3:h:baxter:sigma_spectrum_infusion_syste...
8,CVE-2014-5433,1,2,OR,no,Hardware,Baxter,Wireless Battery Module,16,= 16,,,,,False,175BDC61-5163-4D78-BF02-6B9EF5D38841,cpe:2.3:h:baxter:wireless_battery_module:16:*:...
9,CVE-2014-5434,1,1,OR,no,OS,Baxter,Sigma Spectrum Infusion System Firmware,6.05,= 6.05,,,,,True,E6B02D87-6C75-4D60-8D4C-84628757214F,cpe:2.3:o:baxter:sigma_spectrum_infusion_syste...
